# 🧠 Notebook 1: Load & Visualize Medical Images (DICOM & NIfTI)

**Author:** Motaz Alqaoud, PhD | Senior AI/ML Engineer @ Abbott  
**GitHub:** [motazalqaoud](https://github.com/motazalqaoud)

---

## What you'll learn
- Load DICOM series and extract 3D volumes
- Load NIfTI files (.nii.gz) with their affine matrices
- Understand voxel spacing, orientation, and metadata
- Visualize all 3 anatomical planes (axial, sagittal, coronal)
- Download and use a real public medical dataset

## Why this matters clinically
> A tumor measured as 2.5 cm might actually be 1.8 cm if you ignore voxel spacing.  
> In surgical navigation, a 7mm error can be the difference between clean margins and recurrence.  
> This notebook teaches you to read images the way a clinical AI engineer does — not just loading pixels, but understanding what they represent physically.


## 1. Install & Import Dependencies

In [ ]:
# Install required packages (run once)
# !pip install pydicom nibabel SimpleITK matplotlib numpy

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, os.path.join('..', 'src'))

print("✅ Imports successful")
print(f"NumPy version: {np.__version__}")


## 2. Download a Public Medical Dataset

We'll use the **Medical Segmentation Decathlon** Task06 (Lung) — freely available. Or use your own data.

In [ ]:
# Option A: Create synthetic MRI-like data to run the notebook without downloading
# (Replace with real data paths for actual use)

def create_synthetic_mri(shape=(64, 128, 128), n_lesions=2, seed=42):
    """
    Create a synthetic MRI-like volume with ellipsoidal 'lesions'.
    Useful for testing pipelines without real patient data.
    """
    np.random.seed(seed)
    volume = np.random.normal(0.2, 0.05, shape).astype(np.float32)
    
    # Simulate tissue layers (approximate MRI contrast)
    z, y, x = np.mgrid[0:shape[0], 0:shape[1], 0:shape[2]]
    
    # Outer tissue shell
    sphere = ((z - shape[0]//2)**2 / (shape[0]//2.2)**2 +
              (y - shape[1]//2)**2 / (shape[1]//2.2)**2 +
              (x - shape[2]//2)**2 / (shape[2]//2.2)**2)
    volume[sphere < 1] += 0.4  # Brighter interior
    
    # Add 'lesions' — ellipsoids with higher intensity
    mask = np.zeros(shape, dtype=np.uint8)
    lesion_centers = [(shape[0]//2, shape[1]//3, shape[2]//3),
                      (shape[0]//2, 2*shape[1]//3, 2*shape[2]//3)]
    
    for center in lesion_centers[:n_lesions]:
        cz, cy, cx = center
        r = 12
        lesion = ((z - cz)**2/r**2 + (y - cy)**2/r**2 + (x - cx)**2/r**2) < 1
        volume[lesion] += 0.35
        mask[lesion] = 1
    
    # Clip to [0, 1]
    volume = np.clip(volume, 0, 1)
    return volume, mask

volume_3d, mask_3d = create_synthetic_mri(shape=(64, 128, 128), n_lesions=2)

# Simulate realistic voxel spacing (in mm): z=5mm slices, xy=1mm
spacing = (5.0, 1.0, 1.0)

print(f"Volume shape:   {volume_3d.shape}  (Z, Y, X)")
print(f"Voxel spacing:  {spacing} mm  (Z, Y, X)")
print(f"Physical size:  ({volume_3d.shape[0]*spacing[0]:.0f} x {volume_3d.shape[1]*spacing[1]:.0f} x {volume_3d.shape[2]*spacing[2]:.0f}) mm")
print(f"Intensity range: [{volume_3d.min():.3f}, {volume_3d.max():.3f}]")
print(f"Lesion voxels:  {mask_3d.sum()} / {mask_3d.size} ({100*mask_3d.mean():.2f}% of volume)")


## 3. Use the DICOM & NIfTI Loaders

Below we demonstrate how to use the loaders from `src/preprocessing/` on real data.

In [ ]:
# ── Using the NIfTI loader on a real file ──────────────────────────────────
# Uncomment and set your file path:
# from preprocessing.nifti_loader import load_nifti, reorient_to_ras
# volume, affine, meta = load_nifti("path/to/your/file.nii.gz")
# print(meta)

# ── Using the DICOM loader ──────────────────────────────────────────────────
# from preprocessing.dicom_loader import load_dicom_series
# volume, meta = load_dicom_series("path/to/dicom/folder/")
# print(meta)

# ── For this notebook, we use our synthetic volume ──────────────────────────
meta = {
    'spacing': spacing,
    'shape': volume_3d.shape,
    'modality': 'MRI (synthetic)',
    'orientation': 'RAS',
    'physical_size_mm': tuple(s * sp for s, sp in zip(volume_3d.shape, spacing))
}

print("Volume metadata:")
for k, v in meta.items():
    print(f"  {k}: {v}")


## 4. Visualize: 3 Anatomical Planes

Always look at all 3 planes. A finding in one plane may be clearer in another.

In [ ]:
def show_3planes(volume, mask=None, title="Medical Image — 3 Planes", 
                  cmap='gray', mask_alpha=0.4, figsize=(14, 5)):
    z, y, x = volume.shape
    cz, cy, cx = z // 2, y // 2, x // 2

    slices = [
        ("Axial (top-down)",     volume[cz, :, :],  mask[cz, :, :] if mask is not None else None),
        ("Sagittal (side)",      volume[:, :, cx],  mask[:, :, cx] if mask is not None else None),
        ("Coronal (front-back)", volume[:, cy, :],  mask[:, cy, :] if mask is not None else None),
    ]

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)

    for ax, (plane_name, img, msk) in zip(axes, slices):
        ax.imshow(img, cmap=cmap, aspect='equal')
        if msk is not None:
            overlay = np.zeros((*msk.shape, 4))
            overlay[msk > 0] = [1, 0.2, 0.2, mask_alpha]
            ax.imshow(overlay, aspect='equal')
        ax.set_title(plane_name, fontsize=11)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_3planes(volume_3d, mask=mask_3d, title="Synthetic MRI — 3 Anatomical Planes")
print("\n💡 Red overlay = lesion mask")
print("   Notice how the lesion appears differently in each plane.")


## 5. Slice Browser — Step Through the Volume

In [ ]:
def show_slice_grid(volume, mask=None, axis=0, n_slices=12, cmap='gray'):
    """Show evenly-spaced slices along one axis."""
    n_total = volume.shape[axis]
    indices = np.linspace(2, n_total - 3, n_slices, dtype=int)
    
    cols = 4
    rows = (n_slices + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.5))
    axes = axes.flatten()

    axis_names = {0: "Axial", 1: "Coronal", 2: "Sagittal"}

    for i, idx in enumerate(indices):
        slc = np.take(volume, idx, axis=axis)
        msk = np.take(mask, idx, axis=axis) if mask is not None else None

        axes[i].imshow(slc, cmap=cmap)
        if msk is not None:
            ov = np.zeros((*msk.shape, 4))
            ov[msk > 0] = [1, 0.2, 0.2, 0.5]
            axes[i].imshow(ov)
        axes[i].set_title(f"{axis_names[axis]} #{idx}", fontsize=8)
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f"Slice Grid — {axis_names[axis]} Plane", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_slice_grid(volume_3d, mask=mask_3d, axis=0, n_slices=12)


## 6. Intensity Histogram Analysis

Understanding intensity distributions helps you choose the right normalization.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Full volume histogram
axes[0].hist(volume_3d.flatten(), bins=100, color='steelblue', alpha=0.8, edgecolor='none')
axes[0].set_title("Intensity Distribution — Full Volume", fontweight='bold')
axes[0].set_xlabel("Intensity")
axes[0].set_ylabel("Voxel Count")
axes[0].axvline(volume_3d.mean(), color='red', linestyle='--', label=f'Mean: {volume_3d.mean():.3f}')
axes[0].legend()

# Lesion vs background
lesion_vals = volume_3d[mask_3d > 0]
bg_vals     = volume_3d[mask_3d == 0]

axes[1].hist(bg_vals, bins=80, color='steelblue', alpha=0.6, label=f'Background (n={len(bg_vals):,})', density=True)
axes[1].hist(lesion_vals, bins=40, color='tomato', alpha=0.7, label=f'Lesion (n={len(lesion_vals):,})', density=True)
axes[1].set_title("Lesion vs Background Intensities", fontweight='bold')
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Background — mean: {bg_vals.mean():.3f}, std: {bg_vals.std():.3f}")
print(f"Lesion     — mean: {lesion_vals.mean():.3f}, std: {lesion_vals.std():.3f}")
print(f"\nSeparability (Δmean / pooled std): {abs(lesion_vals.mean()-bg_vals.mean()) / ((lesion_vals.std()+bg_vals.std())/2):.2f}")


## 7. Physical Measurements

How to measure real-world tumor dimensions from voxels.

In [ ]:
# ── Compute lesion physical dimensions ─────────────────────────────────────
lesion_indices = np.argwhere(mask_3d > 0)

if len(lesion_indices) > 0:
    min_idx = lesion_indices.min(axis=0)
    max_idx = lesion_indices.max(axis=0)
    extent_voxels = max_idx - min_idx + 1
    
    sz, sy, sx = spacing
    extent_mm = extent_voxels * np.array([sz, sy, sx])
    
    volume_voxels = mask_3d.sum()
    volume_mm3 = volume_voxels * sz * sy * sx
    volume_cm3 = volume_mm3 / 1000

    print("📐 Lesion Physical Measurements")
    print(f"  Bounding box (voxels): {tuple(extent_voxels)}")
    print(f"  Bounding box (mm):     {tuple(extent_mm.round(1))}")
    print(f"  Volume: {volume_voxels} voxels = {volume_mm3:.1f} mm³ = {volume_cm3:.2f} cm³")
    print()
    print("⚠️  Clinical note: These measurements are ONLY valid if voxel spacing")
    print("    is correctly loaded from the DICOM/NIfTI header.")
else:
    print("No lesion found in mask.")


## ✅ Summary

| What you learned | Why it matters |
|---|---|
| DICOM/NIfTI loading | Foundation of any medical AI pipeline |
| Voxel spacing | Required for all physical measurements |
| 3-plane visualization | Standard clinical review |
| Intensity histograms | Guides normalization choice |
| Physical measurements | Tumor sizing for surgical planning |

**Next:** [Notebook 2 — Preprocessing Pipeline](02_preprocessing_pipeline.ipynb)  
*(Normalization, resampling, augmentation)*
